In [13]:
# Install dependencies
%pip install -q --upgrade huggingface-hub>=1.3.0
%pip install -q transformers torch accelerate
%pip install -q langchain langchain-community langchain-core langchain-text-splitters
%pip install -q faiss-cpu sentence-transformers
%pip install -q xarray netcdf4
%pip install -q numpy pandas

In [14]:
import os

# TODO1: set your personal hugging-face token
# -------------------------------------------------------------------------
os.environ["HF_TOKEN"] = "hf_-------------------------"

In [15]:
# Import required libraries
import os
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, asdict
import warnings
warnings.filterwarnings('ignore')

# transformers (Core LLM libraries)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# LangChain (RAG components)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Data processing
import xarray as xr
import numpy as np
import pandas as pd

print("All dependencies imported successfully")

All dependencies imported successfully


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# We use a lightweight instruction-tuned chat model from Hugging Face Hub.
# This model is small enough to run on CPU or Colab GPU.
model_name = "Qwen/Qwen2.5-3B-Instruct"

# TODO2: Load the tokenizer.
# -------------------------------------------------------------------------
# The tokenizer is responsible for converting text into token IDs.
# trust_remote_code=True allows loading model-specific tokenization logic
# defined by the model authors.
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
# TODO3: Load the language model.
# -------------------------------------------------------------------------
# device_map="auto" automatically places the model on available devices (CPU/GPU).
# dtype=torch.float16 reduces memory usage on supported hardware.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("Model and tokenizer loaded successfully.")


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model and tokenizer loaded successfully.


In [17]:
# Messages are represented as a list of dictionaries.
# Each message explicitly specifies a role ("user" or "assistant")
# and the corresponding content.

prompt = "Please introduce yourself in one sentence."
messages = [
    {"role": "user", "content": prompt}
]

# Modern chat models require a chat template to format messages correctly.
# The tokenizer knows how to apply the correct template for this model.
# tokenize=False returns a string prompt instead of token IDs.
# add_generation_prompt=True appends the assistant role marker.
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert the prompt into model input tensors.
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# TODO4: Generate a response from the model.
# -------------------------------------------------------------------------
# max_new_tokens controls the maximum length of the generated response.
# temperature controls randomness.
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# TODO5: Decode only the newly generated tokens. (only decode the new generated tokens, no duplicate tokens)
# -------------------------------------------------------------------------
# We slice the output to remove the original prompt tokens.
generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("Assistant:", response)


Assistant: I am Qwen, an AI assistant created by Alibaba Cloud to provide assistance and information on a wide range of topics.


In [18]:
from typing import List, Dict

# TODO6: Implement Multi-turn Conversation Memory

# In this task, you will implement a minimal multi-turn chat function
# using Hugging Face Transformers.

# You are given:
# - same pretrained instruction-tuned chat model as section1 (model)
# - same tokenizer with a built-in chat template as section1 (toknizer)
# - A `history` list that stores previous messages

# Your function must:
# 1. Add the user input to the conversation history
# 2. Format the full conversation using the model’s chat template
# 3. Run model generation
# 4. Decode only the assistant’s newly generated tokens
# 5. Append the assistant response back into the history
# -------------------------------------------------------------------------
def hf_chat(
    user_input: str,
    history: List[Dict[str, str]]
) -> str:
    """
    Minimal multi-turn chat function using Hugging Face transformers.

    This function appends the user input to history, formats the context,
    generates a response, and updates the history with the assistant's reply.
    """

    # 1. Add the user input to the conversation history
    history.append({"role": "user", "content": user_input})

    # 2. Format the full conversation using the model’s chat template
    # tokenize=False returns the formatted string for the generation prompt
    prompt = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert the prompt into model input tensors and move to the model's device
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 3. Run model generation
    # We use torch.no_grad() to save memory during inference
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,        # Sufficient length for detailed answers
            do_sample=True,            # Enable sampling for conversational variety
            temperature=0.7,           # Balance between creativity and coherence
            top_p=0.9,                 # Nucleus sampling
            pad_token_id=tokenizer.eos_token_id
        )

    # 4. Decode only the assistant’s newly generated tokens
    # We slice the output to exclude the input prompt tokens
    # Logic: $outputs[0][inputs.input_ids.shape[-1]:]$
    input_length = inputs.input_ids.shape[-1]
    new_tokens = outputs[0][input_length:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # 5. Append the assistant response back into the history for the next turn
    history.append({"role": "assistant", "content": response_text})

    return response_text
    """
    Minimal multi-turn chat function using Hugging Face transformers.

    This function demonstrates the core idea behind conversational memory:
    the model itself is stateless, so all context must be explicitly provided
    as a list of messages.

    Args:
        user_input: The user's input text.
        history: A list of message dictionaries with keys {"role", "content"}.

    Returns:
        The assistant's response text.
    """

    pass



In [19]:
# Initialize an empty conversation history.
chat_history = []

# # First user query
# prompt1 = "What is artificial intelligence?"
# response_1 = hf_chat(prompt1, chat_history)

# print("-"*30)
# print("User:", prompt1)
# print("Assistant:", response_1)
# print()

# # Follow-up query that depends on conversation context
# prompt2 = "Please summarize your previous answer in one sentence."
# response_2 = hf_chat(prompt2, chat_history)

# print("-"*30)
# print("User:", prompt2)
# print("Assistant:", response_2)


In [20]:
import os
import json
from typing import List, Dict, Any

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from google.colab import drive
drive.mount('/content/drive/')

CORPUS_PATH = "/content/drive/MyDrive/Colab Notebooks/medical_corpus.jsonl" # set your real path (e.g., Google Drive path)
TOP_K = 3                              # number of retrieved chunks

# Embedding model (CPU-friendly, strong baseline)
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [21]:
def load_jsonl_corpus(path: str) -> List[Document]:
    """
    Load a JSONL corpus where each line is a retrieval chunk.

    Required fields per line:
      - doc_id: unique chunk id
      - text: chunk content

    Any other fields will be stored as Document.metadata.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Corpus file not found: {path}")

    docs: List[Document] = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)

            if "doc_id" not in obj or "text" not in obj:
                raise ValueError(f"Invalid JSONL at line {line_no}: missing doc_id or text.")

            text = obj["text"]
            metadata = {k: v for k, v in obj.items() if k != "text"}
            docs.append(Document(page_content=text, metadata=metadata))

    if not docs:
        raise ValueError("Corpus loaded 0 documents. Check your file path/content.")
    return docs


In [22]:
docs = load_jsonl_corpus(CORPUS_PATH)
print(f"Loaded {len(docs)} JSONL chunks.")
print("Example doc_id:", docs[0].metadata.get("doc_id"))

Loaded 9233 JSONL chunks.
Example doc_id: alzheimers-dementia-planning-for-the-future-ts.pdf_part_0


In [12]:
# TODO7: initialize embeddings
# -------------------------------------------------------------------------
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL_NAME)

# TODO8: Store vectors in FAISS
# -------------------------------------------------------------------------
vectorstore = FAISS.from_documents(docs, embeddings)

# TODO9: Retriever config (k can be tuned)
# -------------------------------------------------------------------------
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

print("Vector store ready.")


/tmp/ipykernel_707/1795587301.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store ready.


In [23]:
def show_retrieval_evidence(query: str, retrieved_docs: List[Document], max_chars: int = 300) -> None:
    """
    REQUIRED for grading: show what was retrieved.
    """
    print("=" * 90)
    print("RAG EVIDENCE (REQUIRED)")
    print("=" * 90)
    print(f"Query: {query}")
    print(f"Retrieved: {len(retrieved_docs)} chunks\n")

    for i, d in enumerate(retrieved_docs, 1):
        doc_id = d.metadata.get("doc_id", "N/A")
        dtype  = d.metadata.get("type", "N/A")
        var    = d.metadata.get("var", "")
        title  = d.metadata.get("title", "")
        snippet = d.page_content[:max_chars].strip() + ("..." if len(d.page_content) > max_chars else "")

        print(f"[{i}] doc_id={doc_id} | type={dtype} | var={var} | title={title}")
        print(f"Snippet: {snippet}\n")


In [24]:
# 1. Retrieve
def retrieve(query: str, k: int = TOP_K) -> List[Document]:
    """
    Retriever step: return top-k relevant chunks for the query.
    """
    # NOTE: retriever already has default k, but we keep k here for clarity/experimentation.
    return retriever.invoke(query)

In [25]:
# 2. Build Context (with doc_id headers)
def build_rag_messages(context_docs: List[Document], question: str) -> List[Dict[str, str]]:
    """
    Context Builder step:
    Format retrieved chunks into a grounded prompt with explicit [doc_id] headers.
    The model must cite doc_ids in the final answer.
    """
    ctx_blocks = []
    for d in context_docs:
        doc_id = d.metadata.get("doc_id", "unknown")
        ctx_blocks.append(f"[{doc_id}]\n{d.page_content.strip()}")

    context = "\n\n".join(ctx_blocks)

    system = (
        "You are a retrieval-augmented assistant.\n"
        "Answer the user's question using ONLY the provided context.\n"
        "If the context is insufficient, say: \"I don't have enough information from the provided documents.\"\n"
        "You MUST cite sources using doc_id in square brackets, e.g., [var_t2m], [dataset_overview].\n"
        "Do NOT cite anything that is not in the context."
    )

    user = f"Context:\n{context}\n\nQuestion: {question}"

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]



In [26]:
# 3. Generate (LLM call using chat template)
def hf_generate_from_messages(
    messages: List[Dict[str, str]],
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    top_p: float = 0.95,
    top_k: int = 50,
) -> str:
    """
    Hugging Face chat generation helper.
    - If temperature == 0: deterministic decoding (no sampling) and NO sampling params passed.
    - If temperature  > 0: sampling enabled with temperature/top_p/top_k.
    """
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    do_sample = temperature > 0.0

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Only pass sampling params when sampling is enabled (avoids warnings)
    if do_sample:
        gen_kwargs.update(dict(temperature=temperature, top_p=top_p, top_k=top_k))

    with torch.no_grad():
        outputs = model.generate(**gen_kwargs)

    text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()
    return text

In [27]:
def rag_answer(question: str, k: int = TOP_K, show_evidence: bool = True) -> Dict[str, Any]:
    """
    End-to-end RAG:

      1) Retrieve top-k chunks
      2) (Required) display evidence
      3) Build grounded prompt with [doc_id] headers
      4) Generate answer with citations

    Returns:
      {
        "answer": str,
        "evidence": [{"doc_id":..., "title":..., "var":..., "text":...}, ...]
      }
    """

    # TODO10: retrieve related knowledge based on user prompt(only one line)
    # -------------------------------------------------------------------------
    retrieved_docs = retrieve(question, k=k)

    if show_evidence:
        show_retrieval_evidence(question, retrieved_docs)


    # TODO11: build prompt with retrieved knowledge(only one line)
    # -------------------------------------------------------------------------
    messages = build_rag_messages(retrieved_docs, question)

    # TODO12: generate answers with built messages(only one line)
    # -------------------------------------------------------------------------
    answer = hf_generate_from_messages(messages)

    evidence = []
    for d in retrieved_docs:
        evidence.append({
            "doc_id": d.metadata.get("doc_id", "N/A"),
            "title": d.metadata.get("title", "N/A"),
            "var": d.metadata.get("var", ""),
            "text": d.page_content,
        })

    return {"answer": answer, "evidence": evidence}


In [28]:
import re
import json
from typing import Dict, Any

# 假设这个函数已经在之前的格子 (Section 3) 里被定义了
# 如果运行时报错说找不到 hf_generate_from_messages，请确保 Section 3 的那个格子已经运行过了
def smart_speaker_diarization(raw_text: str) -> str:
    """
    使用 Qwen 自动为混乱的对话文本打上标签 (Interviewer: / Patient:)。
    """
    sys_prompt = """
    You are a medical transcript parser. Your job is to format raw dialogue.
    If the input looks like a conversation between a medical professional (or caregiver) and a patient, but lacks speaker labels, you MUST add 'Interviewer:' and 'Patient:' at the beginning of each turn.
    If the input is just a single statement, or already has labels, output it exactly as it is.
    Do NOT add any other text, explanations, or markdown formatting. ONLY output the formatted dialogue.

    Example Input: Hello Mrs Smith how are you today I don't know where my room is
    Example Output:
    Interviewer: Hello Mrs Smith how are you today?
    Patient: I don't know where my room is.
    """

    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Raw text: {raw_text}"}
    ]

    try:
        # 调用之前写好的大模型生成函数
        formatted_text = hf_generate_from_messages(messages, max_new_tokens=500, temperature=0.1)
        # 去掉大模型可能加上的无用 markdown 标记 (比如 ```markdown )
        formatted_text = formatted_text.replace("```", "").strip()
        return formatted_text
    except Exception as e:
        print(f"[Warning] Diarization error: {e}. Falling back to raw text.")
        return raw_text # 失败则退回原文，保证程序不崩溃

def analyze_dementia_risk(text: str) -> Dict[str, Any]:
    """
    包含智能说话者分离功能的认知障碍风险评估工具。
    """
    # ---------------------------------------------------------
    # 1. 智能对话分离 (Speaker Diarization)
    # ---------------------------------------------------------
    # 如果文本包含典型的标签，就直接用；如果没有，就用大模型去“抢救”一下
    if not re.search(r'(?i)(patient|doctor|interviewer|caregiver):', text):
        formatted_transcript = smart_speaker_diarization(text)
    else:
        formatted_transcript = text

    # ---------------------------------------------------------
    # 2. 对话剥离器 (Transcript Parser)
    # ---------------------------------------------------------
    patient_lines = []
    is_transcript = False

    for line in formatted_transcript.split('\n'):
        line = line.strip()
        if not line:
            continue

        lower_line = line.lower()
        if lower_line.startswith("patient:") or lower_line.startswith("elderly:"):
            # 提取冒号后面的患者原话
            patient_lines.append(line.split(':', 1)[1].strip())
            is_transcript = True
        elif lower_line.startswith("doctor:") or lower_line.startswith("interviewer:") or lower_line.startswith("caregiver:"):
            # 遇到医生的话，跳过
            is_transcript = True
            continue

    if is_transcript and patient_lines:
        analyzed_text = " ".join(patient_lines)
    else:
        analyzed_text = text # 如果提取失败，或者只是一句话，就用原文

    text_lower = analyzed_text.lower()

    # ---------------------------------------------------------
    # 3. 关键词与重复性打分
    # ---------------------------------------------------------
    temporal_words = ["what day", "what year", "what time", "is it morning", "is it night"]
    spatial_words = ["where am i", "how to get home", "want to go home", "lost", "where is this"]
    memory_words = ["forgot", "forget", "don't remember", "who are you", "where is my", "what was i saying"]

    score = 0
    matched_signs = []

    for word in temporal_words:
        if word in text_lower:
            score += 2
            matched_signs.append("Temporal Confusion")

    for word in spatial_words:
        if word in text_lower:
            score += 3
            matched_signs.append("Spatial Confusion")

    for word in memory_words:
        if word in text_lower:
            score += 2
            matched_signs.append("Memory Loss")

    sentences = [s.strip() for s in re.split(r'[.?!]', text_lower) if len(s.strip()) > 5]
    if len(sentences) != len(set(sentences)):
        score += 3
        matched_signs.append("Repetitive Speech")

    # ---------------------------------------------------------
    # 4. 诊断结果
    # ---------------------------------------------------------
    risk_score = min(10, score)

    if risk_score >= 5:
        assessment = "High Risk: Significant signs of cognitive decline detected. Medical consultation recommended."
    elif risk_score > 0:
        assessment = "Moderate Risk: Some confusion or memory issues detected. Keep monitoring."
    else:
        assessment = "Low Risk: No obvious linguistic signs of cognitive decline in this text."

    return {
        "original_input": text,
        "llm_formatted_transcript": formatted_transcript, # 方便你打印出来看看大模型整理得对不对
        "analyzed_patient_text": analyzed_text,
        "risk_score_out_of_10": risk_score,
        "detected_signs": list(set(matched_signs)),
        "assessment": assessment
    }

# 更新工具箱注册表
TOOL_REGISTRY = {
    "rag": rag_answer,  # 确保 rag_answer 在这之前被定义过
    "analyze_dementia_risk": analyze_dementia_risk
}

In [29]:
print(analyze_dementia_risk("What day is it today? I forgot. What day is it today?"))

{'original_input': 'What day is it today? I forgot. What day is it today?', 'llm_formatted_transcript': 'Interviewer: What day is it today? I forgot.\nPatient: What day is it today?', 'analyzed_patient_text': 'What day is it today?', 'risk_score_out_of_10': 2, 'detected_signs': ['Temporal Confusion'], 'assessment': 'Moderate Risk: Some confusion or memory issues detected. Keep monitoring.'}


In [30]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
# import xarray as xr
# import numpy as np
# import os
# from typing import Dict, Any

# # NetCDF files (ERA5 daily data) modify to adapt your pathes
# DATA_PATHS = {
#     "t2m": "/content/drive/MyDrive/Colab Notebooks/6895_Assignment1/data/t2m.nc",   # 2m temperature
#     "d2m": "/content/drive/MyDrive/Colab Notebooks/6895_Assignment1/data/d2m.nc",   # 2m dewpoint
#     "u10": "/content/drive/MyDrive/Colab Notebooks/6895_Assignment1/data/u10.nc",   # 10m u wind
#     "v10": "/content/drive/MyDrive/Colab Notebooks/6895_Assignment1/data/v10.nc",   # 10m v wind
#     "msl": "/content/drive/MyDrive/Colab Notebooks/6895_Assignment1/data/msl.nc",   # mean sea level pressure
#     "tp": "/content/drive/MyDrive/Colab Notebooks/6895_Assignment1/data/tp.nc",    # total precipitation
# }

# datasets = {
#     var: xr.open_dataset(path,engine="h5netcdf")
#     for var, path in DATA_PATHS.items()
# }


In [32]:
# def _standardize_da(da: xr.DataArray) -> xr.DataArray:
#     """
#     Standardize ERA5 DataArray to use:
#       - time, lat, lon
#     """
#     rename = {}
#     if "valid_time" in da.dims: rename["valid_time"] = "time"
#     if "latitude" in da.dims: rename["latitude"] = "lat"
#     if "longitude" in da.dims: rename["longitude"] = "lon"
#     if rename:
#         da = da.rename(rename)

#     if "number" in da.dims:
#         da = da.mean(dim="number")

#     return da


In [33]:




# # Tool 1
# def inspect_dataset(variable: str) -> Dict[str, Any]:
#     """
#     Tool: Inspect dataset metadata and time coverage.
#     """
#     if variable not in datasets:
#         return {"error": f"unknown variable '{variable}'"}

#     ds = datasets[variable]
#     var_name = list(ds.data_vars)[0]
#     da = _standardize_da(ds[var_name])

#     return {
#         "variable": variable,
#         "dims": list(da.dims),
#         "shape": tuple(da.shape),
#         "time_start": str(da["time"].values[0]),
#         "time_end": str(da["time"].values[-1]),
#         "units": da.attrs.get("units", "unknown"),
#     }


In [34]:
# # Tool 2
# def compute_stat(
#     variable: str,
#     metric: str = "mean",
#     spatial: str = "box_mean",
#     lat: float = None,
#     lon: float = None,
#     units: str = None,
# ) -> Dict[str, Any]:
#     """
#     Tool: Compute a statistic over climate data.
#     """
#     if variable not in datasets:
#         return {"error": f"unknown variable '{variable}'"}

#     ds = datasets[variable]
#     var_name = list(ds.data_vars)[0]
#     da = _standardize_da(ds[var_name])

#     # spatial reduction
#     if spatial == "box_mean":
#         if "lat" in da.dims: da = da.mean(dim="lat")
#         if "lon" in da.dims: da = da.mean(dim="lon")

#     # metric
#     if metric == "mean":
#         value = float(da.mean())
#     elif metric == "max":
#         value = float(da.max())
#     elif metric == "min":
#         value = float(da.min())
#     elif metric == "sum":
#         value = float(da.sum())
#     else:
#         return {"error": f"unsupported metric '{metric}'"}

#     unit_out = da.attrs.get("units", "native")

#     # temperature conversion
#     if variable in ["t2m", "d2m"] and units == "C":
#         value -= 273.15
#         unit_out = "°C"

#     # precipitation convenience
#     if variable == "tp":
#         return {
#             "variable": variable,
#             "metric": metric,
#             "value_m": value,
#             "value_mm": value * 1000.0,
#             "unit": "mm",
#         }

#     return {
#         "variable": variable,
#         "metric": metric,
#         "value": value,
#         "unit": unit_out,
#     }


In [35]:
# # Tool Test
# print(compute_stat("t2m", metric="mean", units="C"))
# print(compute_stat("tp", metric="sum"))

In [36]:
import json

# 定义我们现在允许的工具
ALLOWED_TOOLS = ["rag", "analyze_dementia_risk"]

def make_router_prompt(question: str) -> str:
    return f"""
    You are a helpful AI assistant for 'MindKeeper', an Alzheimer's care agent.

    Your goal is to decide which tool to use based on the user's input.

    Return ONLY valid JSON with this schema:
    {{
      "plan": [
        {{"tool": "...", "args": {{...}}}}
      ]
    }}

    Available tools:

    1) rag
       - Use this when the user asks for INFORMATION, ADVICE, or MEDICAL KNOWLEDGE.
       - Examples: "What are early symptoms?", "How to handle aggression?", "What is dementia?"
       - args: {{"question": string, "k": integer}}

    2) analyze_dementia_risk
       - Use this when the user is simply CHATTING, talking about their day, or showing signs of CONFUSION.
       - Use this to detect cognitive decline in daily conversation.
       - Examples: "I forgot where I am.", "What day is it?", "I want to go home.", "The weather is nice."
       - args: {{"text": string}}

    Routing rules:
    - If the user asks a question about the disease or caregiving -> Use 'rag'.
    - If the user is speaking like a patient (repeating, confused) or just chatting -> Use 'analyze_dementia_risk'.
    - You can use both if needed, but usually one is enough.

    User input:
    {question}

    JSON:
    """.strip()

In [37]:
import json, re

# TODO13: generate answers with built messages(only one line)
# The router LLM is instructed to output JSON, but in practice the output may contain:
# - Markdown code fences (```json)
# - Extra text before or after the JSON
# - Line breaks or whitespace

# Your task is to extract the first valid JSON object `{...}` from the model output
# and return it as a Python dictionary.

# Requirements:
# - Input: raw string output from the LLM
# - Output: parsed Python dict
# - If no valid JSON object is found, raise an exception
# -------------------------------------------------------------------------
def extract_json(text: str) -> dict:
    """
    Robust JSON extraction: find the first {...} block and parse it.
    """
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        raise ValueError("No valid JSON object found in LLM output.")

    # 2. Extract the string and parse it into a Python dictionary
    json_str = match.group(0)
    return json.loads(json_str)


def route_question(question: str) -> Dict[str, Any]:
    router_text = make_router_prompt(question)

    messages = [
        {"role": "system", "content": "Output strictly valid JSON only. No extra text."},
        {"role": "user", "content": router_text},
    ]

    raw = hf_generate_from_messages(
        messages, max_new_tokens=256, temperature=0.0
    )

    try:
        plan = extract_json(raw)
        if "plan" not in plan or not isinstance(plan["plan"], list) or len(plan["plan"]) == 0:
            raise ValueError("Invalid plan schema")
        return plan
    except Exception:
        # Conservative fallback: RAG-only
        return {"plan": [{"tool": "rag", "args": {"question": question, "k": 3}}]}




In [38]:
# 1. 在这里直接强制绑定我们的花名册 (确保它在执行前绝对存在)
TOOL_REGISTRY = {
    "rag": rag_answer,
    "analyze_dementia_risk": analyze_dementia_risk
}

# 2. 执行计划的函数
def run_plan(plan: Dict[str, Any]) -> List[Dict[str, Any]]:
    trace = []

    # 遍历计划列表中的每一步
    for step in plan.get("plan", []):
        tool_name = step.get("tool")
        tool_args = step.get("args", {})

        # 检查工具是否在上面的花名册里
        if tool_name in TOOL_REGISTRY:
            # 取出工具函数
            tool_func = TOOL_REGISTRY[tool_name]

            # 运行工具
            try:
                result = tool_func(**tool_args)
            except Exception as e:
                result = {"error": str(e)}

            # 记录结果
            trace.append({
                "tool": tool_name,
                "args": tool_args,
                "result": result
            })
        else:
            # 如果找不到工具，记录一个错误提示
            trace.append({
                "tool": tool_name,
                "args": tool_args,
                "result": f"Error: Tool '{tool_name}' not found in TOOL_REGISTRY."
            })

    return trace

In [39]:
def synthesize_answer(question: str, trace: List[Dict[str, Any]]) -> str:
    """
    Synthesize the final answer based on the user question and the tool execution trace.
    """
    trace_str = json.dumps(trace, indent=2, ensure_ascii=False)

    # 我们给“嘴巴”设定全新的人设和说话规则
    sys_msg = "You are MindKeeper, an empathetic and professional AI assistant for Alzheimer's and dementia care."

    user_msg = f"""
    User input: "{question}"

    Here is the data returned by your internal tools:
    {trace_str}

    Instructions for your response:
    1. If the tool used was 'analyze_dementia_risk', politely explain the tool's findings to the user/caregiver. Mention the "risk_score_out_of_10", the "detected_signs", and the "assessment". Speak gently, professionally, and show empathy.
    2. If the tool used was 'rag', provide a helpful answer based on the retrieved information and include the source in brackets.

    Write your final response now:
    """.strip()

    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg}
    ]

    # 调用大模型生成最终回答
    answer = hf_generate_from_messages(messages)

    return answer

In [40]:
def run_agent(question: str) -> Dict[str, Any]:
    plan = route_question(question)
    trace = run_plan(plan)
    final_answer = synthesize_answer(question, trace)
    return {
        "question": question,
        "plan": plan["plan"],
        "trace": trace,
        "final_answer": final_answer,
    }


In [42]:
print("=== MINDKEEPER AGENT INTEGRATION TESTS ===\n")

# 测试 5: 完整的医患对话文本
multi_turn_dialogue = """
Hello Mrs. Smith, how are you feeling today?
I don't remember where my room is.
You are at the clinic. Do you remember what day it is?
What day is it? I forgot. I want to go home.
It's Tuesday.
I want to go home. Where is my mother?
"""

# 把所有的测试题放在一个列表里
agent_questions = [
    # 测试 1: 这是一个科普问题 -> 应该调用 RAG
    "What are the early signs of Alzheimer's disease?",

    # 测试 2: 这是一个家属求助问题 -> 应该调用 RAG
    "My father is becoming aggressive and angry, what should I do?",

    # 测试 3: 这是一个明显的病患对话 (高风险) -> 应该调用 analyze_dementia_risk
    "Where is my mother? I want to go home. Where is my mother?",

    # 测试 4: 这是一个日常对话 (低风险) -> 应该调用 analyze_dementia_risk
    "I had a very nice breakfast today with my daughter.",

    # 测试 5: 真实多轮对话测试 (终极难度) -> 应该调用 analyze_dementia_risk 并成功剥离医生的话
    multi_turn_dialogue
]

for i, q in enumerate(agent_questions, 1):
    print("=" * 90)
    print(f"Test {i} User Input:\n{q.strip()}")

    # 运行 Agent
    result = run_agent(q)

    # 打印它怎么思考的 (Plan)
    print("\n[AI Brain] Router Plan:")
    print(json.dumps(result["plan"], indent=2))

    # 打印它最后怎么说的 (Final Answer)
    print("\n[AI Output] Final Answer:")
    print(result["final_answer"])

    print("-" * 90)

print("\nMindKeeper Integration tests passed. 🎉")

=== MINDKEEPER AGENT INTEGRATION TESTS ===

Test 1 User Input:
What are the early signs of Alzheimer's disease?
RAG EVIDENCE (REQUIRED)
Query: What are the early signs of Alzheimer's disease?
Retrieved: 3 chunks

[1] doc_id=alzheimers-dementia-living-with-alzheimers-ts.pdf_part_16 | type=N/A | var= | title=alzheimers-dementia-living-with-alzheimers-ts.pdf
Snippet: to
 
as
 
early-stage
 
Alzheimer’s.
 
You
 
will
 
start
 
to
 
experience
 
symptoms
 
that
 
interfere
 
with
 
some
 
daily
 
activities.
 
While
 
you
 
will
 
still
 
be
 
able
 
to
 
perform
 
many
 
daily
 
routines,
 
these
 
tasks
 
may
 
become
 
more
 
difficult
 
over
 
time.
 
Friends,...

[2] doc_id=alzheimers-dementia-living-with-alzheimers-ts (1).pdf_part_16 | type=N/A | var= | title=alzheimers-dementia-living-with-alzheimers-ts (1).pdf
Snippet: to
 
as
 
early-stage
 
Alzheimer’s.
 
You
 
will
 
start
 
to
 
experience
 
symptoms
 
that
 
interfere
 
with
 
some
 
daily
 
activities.
 
While
 
you
 
will
 
s